# Latent Compression


## Load SAM Audio and Processor

In [1]:
import torch
from sam_audio import SAMAudio, SAMAudioProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "facebook/sam-audio-base"  # base model

processor = SAMAudioProcessor.from_pretrained(MODEL_ID)
base_model = SAMAudio.from_pretrained(MODEL_ID)
base_model = base_model.to(device=device, dtype=torch.bfloat16)

base_model.eval()  # Freeze model

/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.11.0+cu130)
    Python  3.10.19 (you have 3.11.15)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available

SAMAudio(
  (audio_codec): DACVAE(
    (encoder): Encoder(
      (block): Sequential(
        (0): NormConv1d(1, 64, kernel_size=(7,), stride=(1,), padding=(3,))
        (1): EncoderBlock(
          (block): Sequential(
            (0): ResidualUnit(
              (block): Sequential(
                (0): Snake1d()
                (1): NormConv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(3,))
                (2): Snake1d()
                (3): NormConv1d(64, 64, kernel_size=(1,), stride=(1,))
              )
            )
            (1): ResidualUnit(
              (block): Sequential(
                (0): Snake1d()
                (1): NormConv1d(64, 64, kernel_size=(7,), stride=(1,), padding=(9,), dilation=(3,))
                (2): Snake1d()
                (3): NormConv1d(64, 64, kernel_size=(1,), stride=(1,))
              )
            )
            (2): ResidualUnit(
              (block): Sequential(
                (0): Snake1d()
                (1): NormConv1d(64, 64, 

## Load Data

In [6]:
from anatolian_sam.mixed_dataloader import get_audio_dataloader

target_sr = int(processor.audio_sampling_rate)

dataset, eval_loader = get_audio_dataloader(
    jsonl_path="../data/mixed_metadata_tr.jsonl",
    # data_base_path="/teamspace/studios/turkish-music/anatolian-SAM/data/mixed",
    data_base_path="../data/mixed",
    target_sr=target_sr,
    batch_size=4,
    shuffle=False,
)

Loaded 720 audio tuples from mixed_metadata_tr.jsonl


In [7]:
# Load frozen builtin VAE model
vae = base_model.audio_codec

vae.to(device=device)
vae.eval()  # Freeze VAE
for param in vae.parameters():
    param.requires_grad = False

In [8]:
import os

output_dir = "../data/latents"
os.makedirs(output_dir, exist_ok=True)

## Latent Encode

In [9]:
from tqdm.autonotebook import tqdm


def latent_encode_save(wavs, filenames):
    # Move the padded tensor batch to GPU
    wavs = wavs.to(device)

    with torch.no_grad():
        # Encode the entire batch at once
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            encoded = vae(wavs)

    # Unpack the batch to save files individually
    for i in range(len(filenames)):
        base_name = filenames[i].replace(".wav", ".pt")
        out_path = os.path.join(output_dir, base_name)

        # Extract the specific latent tensor for this item [C, Latent_T]
        single_encoded = encoded[i]

        # Save to disk
        torch.save(single_encoded.cpu(), out_path)


for batch in tqdm(eval_loader):
    latent_encode_save(batch["mixture_wavs"], batch["mixture_filenames"])
    latent_encode_save(batch["target_wavs"], batch["target_filenames"])

print("Latent encoding complete!")

100%|██████████| 180/180 [01:15<00:00,  2.40it/s]

Latent encoding complete!
